In [6]:
# =============================================================================
# 106 — TCGA Methylation Download Validation
# Imports, project-root detection, and input paths
# =============================================================================

import hashlib
import json
import sys
import time
from pathlib import Path

import pandas as pd


# -----------------------------------------------------------------------------
# Detect project root
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "pancancer-epigenetics":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not locate the pancancer-epigenetics project root."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# -----------------------------------------------------------------------------
# Project utilities
# -----------------------------------------------------------------------------

from src.utils.paths import Paths
from src.utils.file_checks import (
    calculate_sha256,
    get_file_size_mb,
    to_project_relative_posix_path,
)


# -----------------------------------------------------------------------------
# Frozen cohort inputs
# -----------------------------------------------------------------------------

METHYLATION_MANIFEST_DIR = (
    Paths.config / "manifests" / "tcga_methylation"
)

FROZEN_MANIFEST_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.txt"
)

FROZEN_FILE_INDEX_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_file_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv"
)

FROZEN_COHORT_METADATA_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_cohort_freeze_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.json"
)


# -----------------------------------------------------------------------------
# Local download directory
# -----------------------------------------------------------------------------

METHYLATION_DOWNLOAD_DIR = (
    Paths.tcga / "methylation"
)


# -----------------------------------------------------------------------------
# Expected frozen-cohort properties
# -----------------------------------------------------------------------------

EXPECTED_FILE_COUNT = 10_897
EXPECTED_TOTAL_SIZE_BYTES = 114_606_561_259

EXPECTED_PLATFORMS = {
    "Illumina Human Methylation 27",
    "Illumina Human Methylation 450",
}


print(f"Project root:          {PROJECT_ROOT}")
print(f"Frozen manifest:       {FROZEN_MANIFEST_PATH}")
print(f"Frozen file index:     {FROZEN_FILE_INDEX_PATH}")
print(f"Frozen cohort record:  {FROZEN_COHORT_METADATA_PATH}")
print(f"Download directory:    {METHYLATION_DOWNLOAD_DIR}")
print()
print(f"Expected payloads:     {EXPECTED_FILE_COUNT:,}")
print(
    f"Expected payload size: "
    f"{EXPECTED_TOTAL_SIZE_BYTES / (1024**3):,.6f} GiB"
)

Project root:          C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics
Frozen manifest:       C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.txt
Frozen file index:     C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_file_index_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.tsv
Frozen cohort record:  C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_cohort_freeze_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450.json
Download directory:    C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\data\raw\tcga\methylation

Expected payloads:     10,897
Expected payload size: 106.735678 GiB


In [7]:
# =============================================================================
# Load and validate frozen cohort artifacts
# =============================================================================

required_input_paths = {
    "manifest": FROZEN_MANIFEST_PATH,
    "file_index": FROZEN_FILE_INDEX_PATH,
    "cohort_freeze": FROZEN_COHORT_METADATA_PATH,
}

missing_inputs = [
    path
    for path in required_input_paths.values()
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing required frozen-cohort inputs:\n"
        + "\n".join(str(path) for path in missing_inputs)
    )

non_file_inputs = [
    path
    for path in required_input_paths.values()
    if not path.is_file()
]

if non_file_inputs:
    raise ValueError(
        "The following frozen-cohort inputs are not files:\n"
        + "\n".join(str(path) for path in non_file_inputs)
    )

if not METHYLATION_DOWNLOAD_DIR.exists():
    raise FileNotFoundError(
        f"Download directory not found: {METHYLATION_DOWNLOAD_DIR}"
    )

if not METHYLATION_DOWNLOAD_DIR.is_dir():
    raise NotADirectoryError(
        f"Download path is not a directory: "
        f"{METHYLATION_DOWNLOAD_DIR}"
    )


# -----------------------------------------------------------------------------
# Load frozen artifacts
# -----------------------------------------------------------------------------

manifest_df = pd.read_csv(
    FROZEN_MANIFEST_PATH,
    sep="\t",
    dtype={
        "id": "string",
        "filename": "string",
        "md5": "string",
        "state": "string",
    },
)

manifest_df["size"] = pd.to_numeric(
    manifest_df["size"],
    errors="raise",
).astype("int64")

file_index_df = pd.read_csv(
    FROZEN_FILE_INDEX_PATH,
    sep="\t",
    dtype={
        "file_id": "string",
        "file_name": "string",
        "md5": "string",
        "state": "string",
        "platform": "string",
    },
)

file_index_df["file_size_bytes"] = pd.to_numeric(
    file_index_df["file_size_bytes"],
    errors="raise",
).astype("int64")

with FROZEN_COHORT_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    cohort_freeze_record = json.load(handle)


# -----------------------------------------------------------------------------
# Validate required columns
# -----------------------------------------------------------------------------

required_manifest_columns = {
    "id",
    "filename",
    "md5",
    "size",
    "state",
}

required_file_index_columns = {
    "file_id",
    "file_name",
    "md5",
    "file_size_bytes",
    "state",
    "platform",
    "project_id",
    "case_uuid",
    "sample_submitter_id",
    "aliquot_uuid",
}

missing_manifest_columns = (
    required_manifest_columns - set(manifest_df.columns)
)

missing_file_index_columns = (
    required_file_index_columns - set(file_index_df.columns)
)

if missing_manifest_columns:
    raise KeyError(
        "Missing manifest columns: "
        + ", ".join(sorted(missing_manifest_columns))
    )

if missing_file_index_columns:
    raise KeyError(
        "Missing file-index columns: "
        + ", ".join(sorted(missing_file_index_columns))
    )


# -----------------------------------------------------------------------------
# Validate row counts and identifiers
# -----------------------------------------------------------------------------

if len(manifest_df) != EXPECTED_FILE_COUNT:
    raise ValueError(
        f"Unexpected manifest row count: {len(manifest_df):,}"
    )

if len(file_index_df) != EXPECTED_FILE_COUNT:
    raise ValueError(
        f"Unexpected file-index row count: {len(file_index_df):,}"
    )

if manifest_df["id"].duplicated().any():
    raise ValueError("Duplicate file IDs found in frozen manifest.")

if file_index_df["file_id"].duplicated().any():
    raise ValueError("Duplicate file IDs found in frozen file index.")

manifest_file_ids = set(manifest_df["id"])
file_index_file_ids = set(file_index_df["file_id"])

if manifest_file_ids != file_index_file_ids:
    raise ValueError(
        "Manifest and file index contain different file-ID sets."
    )


# -----------------------------------------------------------------------------
# Validate manifest against canonical file index
# -----------------------------------------------------------------------------

manifest_index_comparison = (
    manifest_df
    .rename(
        columns={
            "id": "file_id",
            "filename": "manifest_file_name",
            "md5": "manifest_md5",
            "size": "manifest_file_size_bytes",
            "state": "manifest_state",
        }
    )
    .merge(
        file_index_df[
            [
                "file_id",
                "file_name",
                "md5",
                "file_size_bytes",
                "state",
            ]
        ],
        on="file_id",
        how="inner",
        validate="one_to_one",
    )
)

artifact_mismatches = {
    "file_name": int(
        manifest_index_comparison["manifest_file_name"]
        .ne(manifest_index_comparison["file_name"])
        .sum()
    ),
    "md5": int(
        manifest_index_comparison["manifest_md5"]
        .str.lower()
        .ne(manifest_index_comparison["md5"].str.lower())
        .sum()
    ),
    "file_size_bytes": int(
        manifest_index_comparison["manifest_file_size_bytes"]
        .ne(manifest_index_comparison["file_size_bytes"])
        .sum()
    ),
    "state": int(
        manifest_index_comparison["manifest_state"]
        .ne(manifest_index_comparison["state"])
        .sum()
    ),
}

if any(artifact_mismatches.values()):
    raise ValueError(
        f"Manifest-versus-file-index mismatches: "
        f"{artifact_mismatches}"
    )


# -----------------------------------------------------------------------------
# Validate frozen cohort properties
# -----------------------------------------------------------------------------

observed_total_size_bytes = int(manifest_df["size"].sum())
observed_platforms = set(file_index_df["platform"].dropna().unique())
observed_states = set(manifest_df["state"].dropna().unique())

if observed_total_size_bytes != EXPECTED_TOTAL_SIZE_BYTES:
    raise ValueError(
        "Unexpected frozen payload size:\n"
        f"Observed: {observed_total_size_bytes:,} bytes\n"
        f"Expected: {EXPECTED_TOTAL_SIZE_BYTES:,} bytes"
    )

if observed_platforms != EXPECTED_PLATFORMS:
    raise ValueError(
        f"Unexpected frozen platforms: "
        f"{sorted(observed_platforms)}"
    )

if observed_states != {"released"}:
    raise ValueError(
        f"Unexpected manifest states: {sorted(observed_states)}"
    )


# -----------------------------------------------------------------------------
# Validate cohort-freeze record and artifact hashes
# -----------------------------------------------------------------------------

recorded_counts = cohort_freeze_record["frozen_cohort_counts"]

if recorded_counts["n_files"] != EXPECTED_FILE_COUNT:
    raise ValueError(
        "Cohort-freeze JSON contains an unexpected file count."
    )

if (
    recorded_counts["total_file_size_bytes"]
    != EXPECTED_TOTAL_SIZE_BYTES
):
    raise ValueError(
        "Cohort-freeze JSON contains an unexpected payload size."
    )

recorded_platforms = set(
    cohort_freeze_record[
        "platform_acquisition_decision"
    ]["included_platforms"]
)

if recorded_platforms != EXPECTED_PLATFORMS:
    raise ValueError(
        "Cohort-freeze JSON contains an unexpected platform decision."
    )

current_artifact_sha256 = {
    "manifest": calculate_sha256(FROZEN_MANIFEST_PATH),
    "file_index": calculate_sha256(FROZEN_FILE_INDEX_PATH),
}

recorded_artifact_sha256 = {
    label: cohort_freeze_record["versioned_outputs"][label]["sha256"]
    for label in current_artifact_sha256
}

artifact_hash_matches = {
    label: (
        current_artifact_sha256[label]
        == recorded_artifact_sha256[label]
    )
    for label in current_artifact_sha256
}

if not all(artifact_hash_matches.values()):
    raise ValueError(
        f"Frozen artifact SHA-256 mismatch: "
        f"{artifact_hash_matches}"
    )


print("Frozen methylation cohort artifacts loaded and validated.")
print(f"Manifest rows:       {len(manifest_df):,}")
print(f"File-index rows:     {len(file_index_df):,}")
print(f"Unique file IDs:     {len(manifest_file_ids):,}")
print(f"Platforms:           {sorted(observed_platforms)}")
print(
    f"Expected payload:    "
    f"{observed_total_size_bytes / (1024**3):,.6f} GiB"
)
print(f"Manifest SHA-256:    {artifact_hash_matches['manifest']}")
print(f"File-index SHA-256:  {artifact_hash_matches['file_index']}")
print(f"Download directory:  {METHYLATION_DOWNLOAD_DIR}")

Frozen methylation cohort artifacts loaded and validated.
Manifest rows:       10,897
File-index rows:     10,897
Unique file IDs:     10,897
Platforms:           ['Illumina Human Methylation 27', 'Illumina Human Methylation 450']
Expected payload:    106.735678 GiB
Manifest SHA-256:    True
File-index SHA-256:  True
Download directory:  C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\data\raw\tcga\methylation


In [8]:
# =============================================================================
# Inventory local methylation download structure
# =============================================================================

top_level_entries = list(METHYLATION_DOWNLOAD_DIR.iterdir())

top_level_directories = [
    path
    for path in top_level_entries
    if path.is_dir()
]

top_level_files = [
    path
    for path in top_level_entries
    if path.is_file()
]

other_top_level_entries = [
    path
    for path in top_level_entries
    if not path.is_dir() and not path.is_file()
]


# -----------------------------------------------------------------------------
# Inventory direct files, nested directories, and nested files
# -----------------------------------------------------------------------------

direct_file_rows = []
nested_directory_rows = []
nested_file_rows = []
other_folder_entry_rows = []

for folder_number, folder_path in enumerate(
    sorted(top_level_directories),
    start=1,
):
    folder_id = folder_path.name

    try:
        folder_entries = list(folder_path.iterdir())
    except OSError as error:
        raise OSError(
            f"Could not inspect download folder: {folder_path}"
        ) from error

    for entry_path in folder_entries:
        if entry_path.is_file():
            direct_file_rows.append(
                {
                    "folder_id": folder_id,
                    "file_name": entry_path.name,
                    "file_path": entry_path,
                    "file_size_bytes": entry_path.stat().st_size,
                    "extension": entry_path.suffix.lower(),
                }
            )

        elif entry_path.is_dir():
            nested_directory_rows.append(
                {
                    "parent_folder_id": folder_id,
                    "directory_name": entry_path.name,
                    "directory_path": entry_path,
                }
            )

            for nested_path in entry_path.rglob("*"):
                if nested_path.is_file():
                    nested_file_rows.append(
                        {
                            "parent_folder_id": folder_id,
                            "parent_directory": entry_path.name,
                            "relative_path": nested_path.relative_to(
                                folder_path
                            ).as_posix(),
                            "file_path": nested_path,
                            "file_size_bytes": nested_path.stat().st_size,
                        }
                    )

        else:
            other_folder_entry_rows.append(
                {
                    "folder_id": folder_id,
                    "entry_name": entry_path.name,
                    "entry_path": entry_path,
                }
            )

    if folder_number % 2_000 == 0:
        print(
            f"Inventoried {folder_number:,} / "
            f"{len(top_level_directories):,} directories"
        )


# -----------------------------------------------------------------------------
# Build inventory tables
# -----------------------------------------------------------------------------

direct_files_df = pd.DataFrame(
    direct_file_rows,
    columns=[
        "folder_id",
        "file_name",
        "file_path",
        "file_size_bytes",
        "extension",
    ],
)

nested_directories_df = pd.DataFrame(
    nested_directory_rows,
    columns=[
        "parent_folder_id",
        "directory_name",
        "directory_path",
    ],
)

nested_files_df = pd.DataFrame(
    nested_file_rows,
    columns=[
        "parent_folder_id",
        "parent_directory",
        "relative_path",
        "file_path",
        "file_size_bytes",
    ],
)

other_folder_entries_df = pd.DataFrame(
    other_folder_entry_rows,
    columns=[
        "folder_id",
        "entry_name",
        "entry_path",
    ],
)

observed_folder_ids = {
    path.name
    for path in top_level_directories
}


# -----------------------------------------------------------------------------
# Per-folder distributions
# -----------------------------------------------------------------------------

direct_files_per_folder = (
    direct_files_df
    .groupby("folder_id")
    .size()
    .reindex(sorted(observed_folder_ids), fill_value=0)
)

direct_file_count_distribution = (
    direct_files_per_folder
    .value_counts()
    .sort_index()
    .rename_axis("direct_files_per_folder")
    .reset_index(name="folder_count")
)

nested_directories_per_folder = (
    nested_directories_df
    .groupby("parent_folder_id")
    .size()
    .reindex(sorted(observed_folder_ids), fill_value=0)
)

nested_directory_count_distribution = (
    nested_directories_per_folder
    .value_counts()
    .sort_index()
    .rename_axis("nested_directories_per_folder")
    .reset_index(name="folder_count")
)


# -----------------------------------------------------------------------------
# Inventory summary
# -----------------------------------------------------------------------------

observed_direct_size_bytes = int(
    direct_files_df["file_size_bytes"].sum()
)

observed_nested_size_bytes = int(
    nested_files_df["file_size_bytes"].sum()
)

print()
print("Local methylation download structure inventoried.")
print(f"Top-level directories:    {len(top_level_directories):,}")
print(f"Top-level files:          {len(top_level_files):,}")
print(f"Direct files:             {len(direct_files_df):,}")
print(f"Nested directories:       {len(nested_directories_df):,}")
print(f"Nested files:             {len(nested_files_df):,}")
print(f"Other top-level entries:  {len(other_top_level_entries):,}")
print(f"Other folder entries:     {len(other_folder_entries_df):,}")
print(
    f"Observed direct size:     "
    f"{observed_direct_size_bytes / (1024**3):,.6f} GiB"
)
print(
    f"Observed nested size:     "
    f"{observed_nested_size_bytes / (1024**2):,.3f} MiB"
)

print()
print("Direct files per top-level directory:")
print(direct_file_count_distribution)

print()
print("Nested directories per top-level directory:")
print(nested_directory_count_distribution)

if not nested_directories_df.empty:
    print()
    print("Nested-directory names:")
    print(
        nested_directories_df["directory_name"]
        .value_counts()
        .rename_axis("directory_name")
        .reset_index(name="directory_count")
    )

Inventoried 2,000 / 10,897 directories
Inventoried 4,000 / 10,897 directories
Inventoried 6,000 / 10,897 directories
Inventoried 8,000 / 10,897 directories
Inventoried 10,000 / 10,897 directories

Local methylation download structure inventoried.
Top-level directories:    10,897
Top-level files:          0
Direct files:             12,142
Nested directories:       9,102
Nested files:             9,102
Other top-level entries:  0
Other folder entries:     0
Observed direct size:     106.736174 GiB
Observed nested size:     11.268 MiB

Direct files per top-level directory:
   direct_files_per_folder  folder_count
0                        1          9652
1                        2          1245

Nested directories per top-level directory:
   nested_directories_per_folder  folder_count
0                              0          1795
1                              1          9102

Nested-directory names:
  directory_name  directory_count
0           logs             9102


In [9]:
# =============================================================================
# Compare frozen manifest against local download
# =============================================================================

# -----------------------------------------------------------------------------
# Expected payload table
# -----------------------------------------------------------------------------

expected_payloads_df = (
    manifest_df
    .rename(
        columns={
            "id": "folder_id",
            "filename": "file_name",
            "md5": "expected_md5",
            "size": "expected_size_bytes",
            "state": "expected_state",
        }
    )
    .copy()
)

expected_payloads_df["expected_path"] = (
    expected_payloads_df.apply(
        lambda row: (
            METHYLATION_DOWNLOAD_DIR
            / row["folder_id"]
            / row["file_name"]
        ),
        axis=1,
    )
)

expected_folder_ids = set(expected_payloads_df["folder_id"])


# -----------------------------------------------------------------------------
# Folder-level comparison
# -----------------------------------------------------------------------------

missing_folder_ids = sorted(
    expected_folder_ids - observed_folder_ids
)

unexpected_folder_ids = sorted(
    observed_folder_ids - expected_folder_ids
)


# -----------------------------------------------------------------------------
# Payload-level comparison
# -----------------------------------------------------------------------------

observed_payload_columns = (
    direct_files_df[
        [
            "folder_id",
            "file_name",
            "file_path",
            "file_size_bytes",
        ]
    ]
    .rename(
        columns={
            "file_path": "local_path",
            "file_size_bytes": "local_size_bytes",
        }
    )
)

payload_comparison_df = expected_payloads_df.merge(
    observed_payload_columns,
    on=["folder_id", "file_name"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

missing_payloads_df = (
    payload_comparison_df.loc[
        payload_comparison_df["_merge"] == "left_only"
    ]
    .copy()
)

present_payloads_df = (
    payload_comparison_df.loc[
        payload_comparison_df["_merge"] == "both"
    ]
    .copy()
)

size_mismatches_df = (
    present_payloads_df.loc[
        present_payloads_df["expected_size_bytes"]
        != present_payloads_df["local_size_bytes"]
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Identify auxiliary direct files
# -----------------------------------------------------------------------------

expected_payload_keys = expected_payloads_df[
    ["folder_id", "file_name"]
]

extra_direct_files_df = (
    direct_files_df
    .merge(
        expected_payload_keys,
        on=["folder_id", "file_name"],
        how="left",
        indicator=True,
    )
    .loc[lambda frame: frame["_merge"] == "left_only"]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

annotation_files_df = (
    extra_direct_files_df.loc[
        extra_direct_files_df["file_name"] == "annotations.txt"
    ]
    .copy()
)

unexpected_auxiliary_files_df = (
    extra_direct_files_df.loc[
        extra_direct_files_df["file_name"] != "annotations.txt"
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Detect partial-like or temporary files
# -----------------------------------------------------------------------------

partial_like_suffixes = (
    ".partial",
    ".part",
    ".tmp",
    ".download",
    ".incomplete",
)

direct_partial_like_files_df = (
    direct_files_df.loc[
        direct_files_df["file_name"]
        .str.lower()
        .str.endswith(partial_like_suffixes)
    ]
    .copy()
)

nested_partial_like_files_df = (
    nested_files_df.loc[
        nested_files_df["relative_path"]
        .str.lower()
        .str.endswith(partial_like_suffixes)
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Size summaries
# -----------------------------------------------------------------------------

expected_payload_size_bytes = int(
    expected_payloads_df["expected_size_bytes"].sum()
)

observed_payload_size_bytes = int(
    present_payloads_df["local_size_bytes"].sum()
)

extra_direct_size_bytes = int(
    extra_direct_files_df["file_size_bytes"].sum()
)

annotation_size_bytes = int(
    annotation_files_df["file_size_bytes"].sum()
)


# -----------------------------------------------------------------------------
# Structural validation summary
# -----------------------------------------------------------------------------

print("Expected-versus-observed comparison completed.")
print(f"Expected folders:          {len(expected_folder_ids):,}")
print(f"Observed folders:          {len(observed_folder_ids):,}")
print(f"Missing folders:           {len(missing_folder_ids):,}")
print(f"Unexpected folders:        {len(unexpected_folder_ids):,}")
print(
    f"Expected payloads present: "
    f"{len(present_payloads_df):,}"
)
print(f"Missing payloads:          {len(missing_payloads_df):,}")
print(f"Size mismatches:           {len(size_mismatches_df):,}")
print(f"Extra direct files:        {len(extra_direct_files_df):,}")
print(f"Annotation files:          {len(annotation_files_df):,}")
print(
    f"Unexpected auxiliary files:"
    f" {len(unexpected_auxiliary_files_df):,}"
)
print(
    f"Direct partial-like files: "
    f"{len(direct_partial_like_files_df):,}"
)
print(
    f"Nested partial-like files: "
    f"{len(nested_partial_like_files_df):,}"
)
print(
    f"Expected payload size:     "
    f"{expected_payload_size_bytes / (1024**3):,.6f} GiB"
)
print(
    f"Observed payload size:     "
    f"{observed_payload_size_bytes / (1024**3):,.6f} GiB"
)
print(
    f"Annotation size:           "
    f"{annotation_size_bytes / (1024**2):,.3f} MiB"
)


# -----------------------------------------------------------------------------
# Auxiliary-file summary
# -----------------------------------------------------------------------------

if not extra_direct_files_df.empty:
    print()
    print("Extra direct files by exact filename:")
    print(
        extra_direct_files_df
        .groupby("file_name", as_index=False)
        .agg(
            file_count=("file_name", "size"),
            total_size_bytes=("file_size_bytes", "sum"),
        )
        .assign(
            total_size_mib=lambda frame: (
                frame["total_size_bytes"] / (1024**2)
            )
        )
        .sort_values(
            ["file_count", "file_name"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )


# -----------------------------------------------------------------------------
# Fail closed if acquisition structure is incomplete
# -----------------------------------------------------------------------------

structural_failures = {
    "missing_folders": len(missing_folder_ids),
    "unexpected_folders": len(unexpected_folder_ids),
    "missing_payloads": len(missing_payloads_df),
    "size_mismatches": len(size_mismatches_df),
    "unexpected_auxiliary_files": len(
        unexpected_auxiliary_files_df
    ),
    "direct_partial_like_files": len(
        direct_partial_like_files_df
    ),
    "top_level_files": len(top_level_files),
    "other_top_level_entries": len(other_top_level_entries),
    "other_folder_entries": len(other_folder_entries_df),
}

"""
nonzero_structural_failures = {
    label: count
    for label, count in structural_failures.items()
    if count != 0
}

if observed_payload_size_bytes != expected_payload_size_bytes:
    nonzero_structural_failures[
        "payload_size_difference_bytes"
    ] = (
        observed_payload_size_bytes
        - expected_payload_size_bytes
    )

if nonzero_structural_failures:
    raise RuntimeError(
        "Local download failed structural validation:\n"
        + json.dumps(
            nonzero_structural_failures,
            indent=2,
        )
    )

print()
print("Local download passed structural and size validation.")
"""

Expected-versus-observed comparison completed.
Expected folders:          10,897
Observed folders:          10,897
Missing folders:           0
Unexpected folders:        0
Expected payloads present: 10,897
Missing payloads:          0
Size mismatches:           0
Extra direct files:        1,245
Annotation files:          1,245
Unexpected auxiliary files: 0
Direct partial-like files: 0
Nested partial-like files: 0
Expected payload size:     106.735678 GiB
Observed payload size:     106.735678 GiB
Annotation size:           0.508 MiB

Extra direct files by exact filename:
         file_name  file_count  total_size_bytes  total_size_mib
0  annotations.txt        1245            533151        0.508452


'\nnonzero_structural_failures = {\n    label: count\n    for label, count in structural_failures.items()\n    if count != 0\n}\n\nif observed_payload_size_bytes != expected_payload_size_bytes:\n    nonzero_structural_failures[\n        "payload_size_difference_bytes"\n    ] = (\n        observed_payload_size_bytes\n        - expected_payload_size_bytes\n    )\n\nif nonzero_structural_failures:\n    raise RuntimeError(\n        "Local download failed structural validation:\n"\n        + json.dumps(\n            nonzero_structural_failures,\n            indent=2,\n        )\n    )\n\nprint()\nprint("Local download passed structural and size validation.")\n'

In [10]:
# =============================================================================
# Diagnose partial payloads and build pending-download manifest
# =============================================================================

def split_partial_suffix(file_name):
    """Return expected filename and detected partial suffix."""
    lower_name = file_name.lower()

    for suffix in partial_like_suffixes:
        if lower_name.endswith(suffix):
            return file_name[:-len(suffix)], suffix

    return file_name, None


# -----------------------------------------------------------------------------
# Map partial files to expected payloads
# -----------------------------------------------------------------------------

partial_payloads_df = direct_partial_like_files_df.copy()

partial_name_parts = (
    partial_payloads_df["file_name"]
    .apply(split_partial_suffix)
)

partial_payloads_df["expected_file_name_candidate"] = (
    partial_name_parts.str[0]
)

partial_payloads_df["partial_suffix"] = (
    partial_name_parts.str[1]
)

expected_payload_lookup = (
    expected_payloads_df[
        [
            "folder_id",
            "file_name",
            "expected_size_bytes",
            "expected_md5",
        ]
    ]
    .rename(
        columns={
            "file_name": "expected_file_name_candidate",
        }
    )
)

partial_payload_comparison_df = (
    partial_payloads_df
    .merge(
        expected_payload_lookup,
        on=[
            "folder_id",
            "expected_file_name_candidate",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

matched_partial_payloads_df = (
    partial_payload_comparison_df.loc[
        partial_payload_comparison_df["_merge"] == "both"
    ]
    .copy()
)

unmatched_partial_payloads_df = (
    partial_payload_comparison_df.loc[
        partial_payload_comparison_df["_merge"] == "left_only"
    ]
    .copy()
)

matched_partial_payloads_df["size_matches_expected"] = (
    matched_partial_payloads_df["file_size_bytes"]
    == matched_partial_payloads_df["expected_size_bytes"]
)

partial_size_mismatches_df = (
    matched_partial_payloads_df.loc[
        ~matched_partial_payloads_df["size_matches_expected"]
    ]
    .copy()
)


# -----------------------------------------------------------------------------
# Confirm one-to-one correspondence with missing payloads
# -----------------------------------------------------------------------------

missing_payload_keys = set(
    zip(
        missing_payloads_df["folder_id"],
        missing_payloads_df["file_name"],
    )
)

matched_partial_keys = set(
    zip(
        matched_partial_payloads_df["folder_id"],
        matched_partial_payloads_df[
            "expected_file_name_candidate"
        ],
    )
)

missing_without_partial = (
    missing_payload_keys - matched_partial_keys
)

partial_without_missing_payload = (
    matched_partial_keys - missing_payload_keys
)

if not unmatched_partial_payloads_df.empty:
    raise RuntimeError(
        f"{len(unmatched_partial_payloads_df):,} partial files "
        "could not be linked to expected manifest payloads."
    )

if missing_without_partial:
    raise RuntimeError(
        f"{len(missing_without_partial):,} missing payloads do not "
        "have a corresponding partial file."
    )

if partial_without_missing_payload:
    raise RuntimeError(
        f"{len(partial_without_missing_payload):,} partial files do "
        "not correspond to currently missing payloads."
    )


# -----------------------------------------------------------------------------
# Build operational pending manifest
# -----------------------------------------------------------------------------

pending_file_ids = set(
    missing_payloads_df["folder_id"]
)

pending_manifest_df = (
    manifest_df.loc[
        manifest_df["id"].isin(pending_file_ids)
    ]
    .sort_values("id")
    .reset_index(drop=True)
    .copy()
)

if len(pending_manifest_df) != len(missing_payloads_df):
    raise RuntimeError(
        "Pending manifest row count does not match the number "
        "of missing payloads."
    )

PENDING_MANIFEST_PATH = (
    METHYLATION_MANIFEST_DIR
    / "gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450_probe.txt"
)

temporary_pending_path = PENDING_MANIFEST_PATH.with_name(
    PENDING_MANIFEST_PATH.name + ".tmp"
)

pending_manifest_df.to_csv(
    temporary_pending_path,
    sep="\t",
    index=False,
    lineterminator="\n",
)

temporary_pending_path.replace(PENDING_MANIFEST_PATH)


# -----------------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------------

pending_size_bytes = int(
    pending_manifest_df["size"].sum()
)

partial_size_bytes = int(
    partial_payloads_df["file_size_bytes"].sum()
)

print("Partial-download diagnosis completed.")
print(f"Missing payloads:              {len(missing_payloads_df):,}")
print(f"Matched partial files:         {len(matched_partial_payloads_df):,}")
print(f"Unmatched partial files:       {len(unmatched_partial_payloads_df):,}")
print(f"Partial size mismatches:       {len(partial_size_mismatches_df):,}")
print(f"Missing without partial file:  {len(missing_without_partial):,}")
print(
    f"Pending manifest size:         "
    f"{pending_size_bytes / (1024**3):,.6f} GiB"
)
print(
    f"Current partial-file size:     "
    f"{partial_size_bytes / (1024**3):,.6f} GiB"
)

print()
print("Partial suffixes:")
print(
    partial_payloads_df["partial_suffix"]
    .value_counts(dropna=False)
    .rename_axis("partial_suffix")
    .reset_index(name="file_count")
)

print()
print("Pending manifest written:")
print(PENDING_MANIFEST_PATH)
print()
print(
    "This is an operational recovery manifest and should not "
    "be committed after the download is completed."
)

Partial-download diagnosis completed.
Missing payloads:              0
Matched partial files:         0
Unmatched partial files:       0
Partial size mismatches:       0
Missing without partial file:  0
Pending manifest size:         0.000000 GiB
Current partial-file size:     0.000000 GiB

Partial suffixes:
Empty DataFrame
Columns: [partial_suffix, file_count]
Index: []

Pending manifest written:
C:\Users\paula\OneDrive\Documentos\Proyectos\pancancer-epigenetics\config\manifests\tcga_methylation\gdc_manifest_tcga_primary_tumor_methylation_array_sesame_beta_values_hm27_hm450_probe.txt

This is an operational recovery manifest and should not be committed after the download is completed.
